# Day 15 — Ship it: streaming API, final eval report, design doc
**Theory:** [day-15-ship-it.md](../theory/day-15-ship-it.md) · ⏱ ~90 min · Level: medium

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Start the ShopEase API in the background
In Codespaces you can also run it in a terminal: `cd projects/shopease-support-agent && uvicorn app:app --port 8000`.
The PORTS tab then gives you a URL to share.

In [ ]:
import threading, json, time
import httpx, uvicorn
sys.path.insert(0, str(REPO / "projects" / "shopease-support-agent"))
from app import app

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(50):
    try:
        print(httpx.get("http://127.0.0.1:8000/health").json()); break
    except httpx.ConnectError:
        time.sleep(0.2)

## 2. Stream a reply (Server-Sent Events)
Text arrives token by token; tool calls arrive as status events. This is what keeps users from staring at a spinner.

In [ ]:
def chat(conversation_id, customer_id, message):
    body = {"conversation_id": conversation_id, "customer_id": customer_id, "message": message}
    t0, first = time.time(), None
    with httpx.stream("POST", "http://127.0.0.1:8000/chat", json=body, timeout=120) as r:
        for line in r.iter_lines():
            if not line.startswith("data: "):
                continue
            ev = json.loads(line[6:])
            if ev["type"] == "text":
                first = first or time.time() - t0
                print(ev["text"], end="", flush=True)
            elif ev["type"] == "tool":
                print(f"\n   [checking: {ev['name']}]")
            elif ev["type"] == "route":
                print(f"[route: {ev['route']}]")
            elif ev["type"] == "done":
                print(f"\n\n✅ done · steps={ev['steps']} · ${ev['cost']} · time to first token {first or 0:.1f}s "
                      f"· total {time.time() - t0:.1f}s")

chat("demo-1", "C002", "My shoes O2003 never arrived. Can I get a refund?")

In [ ]:
chat("demo-1", "C002", "OK, please go ahead with the refund.")   # same conversation_id → same history

## 3. Final eval report (the full suite)
⚠️ Runs 20 cases on the real model (roughly a few tens of cents). Add `--repeats 3` for pass^3 when you're ready.

In [ ]:
import subprocess
out = subprocess.run([sys.executable, "projects/shopease-support-agent/evals/run_evals.py"], capture_output=True, text=True)
print(out.stdout[-3000:])
print(pathlib.Path("projects/shopease-support-agent/evals/report.md").read_text())

## 4. Your one-page design doc
Fill this in (it's saved to `week-3/practice/scratch/design-doc.md`). It's your portfolio piece and interview story.

In [ ]:
DOC = """# ShopEase Support Agent: design doc

## Problem & scope
- What the agent does / refuses:
- Human escalation path:

## Architecture
router (Gemini Flash-Lite, structured output) → FAQ (hybrid RAG) | agent loop (Gemini Flash, 8 tools) | human handoff → output guard

## Key decisions (and why)
1. Workflow before agent (router):
2. Business rules in code (eligibility, approvals, idempotency):
3. Single writer, read-only subagents:
4. Caching + context hygiene:

## Results
- Eval pass rate / pass^3:
- Cost per conversation, p50/p95 latency:
- Red-team results:

## Known limitations & next steps
-
"""
p = pathlib.Path("week-3/practice/scratch/design-doc.md"); p.parent.mkdir(parents=True, exist_ok=True)
if not p.exists():
    p.write_text(DOC)
print("edit:", p)

In [ ]:
server.should_exit = True  # stop the API

## 🎓 Done!
Go through the production readiness checklist in the theory note, then read the case studies in
[`case-studies/`](../../case-studies/README.md) to see how frontier labs made the same decisions at scale.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None